<div style="font-family: 'Helvetica Neue', Arial, sans-serif; background:#fff; padding: 36px 40px; border-radius: 8px; margin-bottom: 4px; position:relative; overflow:hidden; border: 1.5px solid #e8e8e8;">

  <div style="font-size:130px; font-weight:900; color:rgba(0,0,0,0.04); position:absolute; top:-20px; right:30px; line-height:1; letter-spacing:-0.05em;">02</div>

  <div style="font-size:10px; color:#00D563; letter-spacing:0.25em; text-transform:uppercase; margin-bottom:16px;">NOVA IMS &middot; 2025/2026</div>
  <div style="font-size:36px; font-weight:800; color:#111; letter-spacing:-0.02em; line-height:1.1; margin-bottom:6px;">One Pipeline, <span style="color:#00D563;">One Model.</span></div>
  <div style="font-size:12px; color:#00D563; font-weight:500; margin-bottom:24px;">Notebook 2 &mdash; Final Solution &middot; Restart &amp; Run All</div>

  <div style="display:flex; gap:48px;">
    <div>
      <div style="font-size:9px; color:#00D563; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:6px;">Group 33</div>
      <div style="font-size:11px; color:#555; line-height:1.9;">Alexandra Varela, 20250514<br>Francisca Fernandes, 20250406<br>Mariana Melo, 20250414<br>Tiago Antunes, 20250357</div>
    </div>
    <div>
      <div style="font-size:9px; color:#00D563; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:6px;">Course</div>
      <div style="font-size:11px; color:#555; line-height:1.9;">Text Mining<br>MSc Data Science &amp; Advanced Analytics<br>NOVA Information Management School</div>
    </div>
  </div>
</div>

**Submitted solution — an integrated weighted soft-voting ensemble pipeline.**
The instructor confirmed (e-mail, June 2026) that an ensemble is an acceptable final solution
provided it is *a single, integrated end-to-end pipeline that consumes the input data in a unified
way through to the final prediction*. This notebook is exactly that: a weighted soft-vote over
**8 fine-tuned transformer encoders**, with weights found by exhaustive out-of-fold grid search
(stored in `results/tables/ensemble_optimal_result.json`).

- **Pipeline:** raw tweets → `fix_tweet` preprocessing → 8 encoders → weighted soft-vote → label.
- **Primary encoder** (`finbert_fintwitter_10ep_fixtext`) is trained live below (10-fold, load-or-train); the other seven members are loaded from committed per-model probabilities and are fully reproducible via `scripts/run_all_10fold.ps1`.
- **Preprocessing:** `fix_tweet` — `ftfy` mojibake repair + truncation-artefact/URL cleanup (alters 50.1% of tweets).
- **Protocol:** 10-fold stratified CV, seed 42, best-checkpoint-per-fold, fp16 + GradScaler, cosine LR warmup, class-weighted loss + label smoothing.

| Configuration | OOF F1-macro |
|---|---|
| Best individual encoder (FinBERT 10ep fix_text) | 0.9082 |
| Distilled single model (extra work, 12ep) | 0.9139 |
| **Weighted soft-vote ensemble, 8 encoders (SUBMITTED)** | **0.9201** |

The ensemble gain over the best individual encoder is statistically significant (paired bootstrap 95% CI of the F1 difference: [+0.0066, +0.0172], n=1000). **Runtime:** ~2 min with the committed probabilities; ~40 min on a CUDA GPU if the primary-encoder cache is absent. **Instructions:** Kernel → Restart Kernel and Run All Cells. Produces `pred_33.csv`.

In [1]:
# Cell 1: Imports and reproducibility
import os, sys, re, time, gc, json, warnings, tempfile
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import ftfy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')
print(f'ftfy {ftfy.__version__}')

Seed fixed: 42
Python: 3.11.9 (tags/v3.11.9 | torch 2.12.0+cu130
ftfy 6.3.1


In [2]:
# Cell 3: Preprocessing — ftfy mojibake fix + truncation cleanup
# Text-quality audit of the source CSV: 7.0% of tweets carry UTF-8 mojibake
# (e.g. â€™ instead of '), 16.4% end in truncation artefacts and 46.8% contain
# URLs with no sentiment value. fix_tweet repairs/removes these (alters 50.1%
# of tweets). Ablation at 10-fold: +0.26pp OOF F1-macro on FinBERT vs raw text.

_TRUNC_RE = re.compile(r'[�…°�…]+\s*(https?://\S*)?$')
_URL_RE    = re.compile(r'https?://\S+')
_TRAIL_RE  = re.compile(r'[\s\-–:]+$')

def fix_tweet(text: str) -> str:
    """Fix mojibake (ftfy) and remove truncation artefacts + bare URLs."""
    text = ftfy.fix_text(str(text))
    text = _TRUNC_RE.sub('', text)
    text = _URL_RE.sub('', text)
    return _TRAIL_RE.sub('', text).strip()

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

texts      = [fix_tweet(t) for t in train['text'].tolist()]
test_texts = [fix_tweet(t) for t in test['text'].tolist()]
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())
print(f'\nSample before: {train["text"].iloc[1][:90]}')
print(f'Sample after:  {texts[1][:90]}')

Train: (9543, 2) | Test: (2388, 2)
Label distribution (0=Bearish, 1=Bullish, 2=Neutral):
label
0    1442
1    1923
2    6178
Name: count, dtype: int64

Sample before: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.c
Sample after:  $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean


In [3]:
# Cell 4: Configuration — weighted soft-voting ensemble (the submitted pipeline)
# Ensemble weights come from an exhaustive out-of-fold grid search
# (results/tables/ensemble_optimal_result.json). The primary encoder is trained
# live below; the other members are loaded from committed per-model probabilities.

PRIMARY_TAG  = 'finbert_fintwitter_10ep_fixtext'   # best individual encoder, trained live
MODEL_NAME   = 'nickmuchi/finbert-tone-finetuned-fintwitter-classification'
MAXLEN       = 128
LR           = 5e-6
WEIGHT_DECAY = 0.01
EPOCHS       = 10
N_FOLDS      = 10
WARMUP_RATIO = 0.06
LABEL_SMOOTH = 0.05
GRAD_ACCUM   = 1

# Ensemble members and their grid-searched weights
ENSEMBLE = json.loads(Path('results/tables/ensemble_optimal_result.json').read_text())
ENSEMBLE_WEIGHTS = ENSEMBLE['models']
print('Weighted soft-vote ensemble — members and weights:')
for t, w in ENSEMBLE_WEIGHTS.items():
    flag = '  <- primary (trained live)' if t == PRIMARY_TAG else ''
    print(f'  w={w:<5} {t}{flag}')
print(f'Cached ensemble OOF F1-macro: {ENSEMBLE["oof_f1_macro"]:.4f}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    BATCH, EVAL_BATCH = 16, 32
    AMP_DTYPE = torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    USE_SCALER = True
    print(f'\nGPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | fp16 + GradScaler')
else:
    BATCH, EVAL_BATCH = 8, 32
    AMP_DTYPE, USE_SCALER = None, False
    torch.set_num_threads(min(12, os.cpu_count()))
    print('\nCPU fallback')

# Inverse-frequency class weights (for the live-trained primary encoder)
counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class weights: {[round(x,3) for x in class_weights.tolist()]}')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

Weighted soft-vote ensemble — members and weights:
  w=1.0   finbert_fintwitter_10ep_fixtext  <- primary (trained live)
  w=1.0   finbert_fintwitter_7ep
  w=1.0   roberta_large_ts_v2
  w=1.0   debertav3_large_6ep
  w=0.75  finbert_fintwitter_10ep
  w=0.75  finbert_fintwitter
  w=0.75  debertav3_large_6ep_fixtext
  w=0.5   deberta_base_finance_fixtext
Cached ensemble OOF F1-macro: 0.9201

GPU: NVIDIA GeForce RTX 5070 | VRAM 12.8 GB | fp16 + GradScaler
class weights: [2.206, 1.654, 0.515]


In [4]:
# Cell 5: Model builder, training loop and inference helpers (for the primary encoder)
# Class-weighted cross-entropy + label smoothing, cosine warmup, fp16 + GradScaler,
# best-checkpoint-per-fold (selects the epoch by validation macro-F1, not the last).

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
        dtype=torch.float32,   # fp32 params required for GradScaler
    ).to(DEVICE)


@torch.no_grad()
def predict_proba(model, txts):
    model.eval()
    out = []
    for i in range(0, len(txts), EVAL_BATCH):
        enc = tok(txts[i:i+EVAL_BATCH], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda' and AMP_DTYPE is not None:
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)


def run_fold(tr_texts, tr_labels, va_texts, va_y, all_test_texts, fold):
    """Train one fold with best-checkpoint selection. Returns (va_proba, test_proba, best_f1)."""
    seed_all(SEED + fold)
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(tr_texts) / BATCH))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * steps_per_epoch * EPOCHS), steps_per_epoch * EPOCHS)
    ce_loss = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=LABEL_SMOOTH)
    scaler = torch.amp.GradScaler('cuda') if USE_SCALER else None

    best_f1, best_va_proba = -1.0, None
    n = len(tr_texts)
    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt = Path(tmpdir) / 'best.pt'
        for epoch in range(EPOCHS):
            model.train()
            order = np.random.permutation(n)
            running, t0 = 0.0, time.time()
            for i in range(0, n, BATCH):
                bidx = order[i:i+BATCH]
                bt = [tr_texts[j] for j in bidx]
                bl = torch.tensor([tr_labels[j] for j in bidx], dtype=torch.long, device=DEVICE)
                enc = tok(bt, padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                optimizer.zero_grad(set_to_none=True)
                if DEVICE == 'cuda' and AMP_DTYPE is not None:
                    with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                        loss = ce_loss(model(**enc).logits, bl)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                else:
                    loss = ce_loss(model(**enc).logits, bl)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                running += float(loss.item())
            va_proba = predict_proba(model, va_texts)
            ep_f1 = f1_score(va_y, va_proba.argmax(1), average='macro')
            star = ' *** best ***' if ep_f1 > best_f1 else ''
            print(f'    fold {fold+1} epoch {epoch+1}/{EPOCHS} '
                  f'loss={running/steps_per_epoch:.4f} val_f1={ep_f1:.6f} ({time.time()-t0:.0f}s){star}')
            if ep_f1 > best_f1:
                best_f1 = ep_f1
                best_va_proba = va_proba.copy()
                torch.save(model.state_dict(), ckpt)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        best_test_proba = predict_proba(model, all_test_texts)
    print(f'  fold {fold+1} best val macro-F1={best_f1:.6f}')
    return best_va_proba, best_test_proba, best_f1

print('Helpers defined.')

Helpers defined.


In [5]:
# Cell 6: Assemble every ensemble member's probabilities (load-or-train)
# Primary encoder: trained live (10-fold) if its committed cache is absent.
# Other members: loaded from committed prob_test_{tag}.csv / oof_proba_{tag}
# (each fully reproducible by scripts/run_all_10fold.ps1).

member_test = {}   # tag -> (2388, 3) test probabilities
member_oof  = {}   # tag -> (9543, 3) out-of-fold probabilities

def _load_member(tag):
    csv_p = Path(f'results/predictions/prob_test_{tag}.csv')
    npy_p = Path(f'results/predictions/test_proba_{tag}.npy')
    oof_n = Path(f'results/predictions/oof_proba_{tag}.npy')
    oof_c = Path(f'results/predictions/oof_proba_{tag}.csv')
    test_p = (pd.read_csv(csv_p)[['p0','p1','p2']].values if csv_p.exists()
              else np.load(npy_p) if npy_p.exists() else None)
    oof_p = (np.load(oof_n) if oof_n.exists()
             else pd.read_csv(oof_c)[['p0','p1','p2']].values if oof_c.exists() else None)
    return (test_p.astype(np.float32) if test_p is not None else None,
            oof_p.astype(np.float32) if oof_p is not None else None)

# ── primary encoder: load cache or train live ───────────────────────────────
ptest, poof = _load_member(PRIMARY_TAG)
if ptest is not None and poof is not None:
    member_test[PRIMARY_TAG], member_oof[PRIMARY_TAG] = ptest, poof
    print(f'[CACHED]  primary {PRIMARY_TAG}  (test {ptest.shape}, oof {poof.shape})')
else:
    print(f'[TRAINING] primary {PRIMARY_TAG} — {N_FOLDS}-fold from scratch...')
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(y), 3), np.float32)
    tsum = np.zeros((len(test_texts), 3), np.float32)
    t_start = time.time()
    for fold, (tr_idx, va_idx) in enumerate(cv.split(texts, y)):
        print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
        vp, tp, _ = run_fold([texts[i] for i in tr_idx], [int(y[i]) for i in tr_idx],
                             [texts[i] for i in va_idx], y[va_idx], test_texts, fold)
        oof[va_idx] = vp; tsum += tp
        gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    member_test[PRIMARY_TAG] = tsum / N_FOLDS
    member_oof[PRIMARY_TAG] = oof
    Path('results/predictions').mkdir(parents=True, exist_ok=True)
    np.save(f'results/predictions/oof_proba_{PRIMARY_TAG}.npy', oof)
    np.save(f'results/predictions/test_proba_{PRIMARY_TAG}.npy', member_test[PRIMARY_TAG])
    pd.DataFrame(member_test[PRIMARY_TAG], columns=['p0','p1','p2']).to_csv(
        f'results/predictions/prob_test_{PRIMARY_TAG}.csv', index=False)
    print(f'  trained in {(time.time()-t_start)/60:.1f} min; cache saved.')

# ── other members: load committed probabilities ─────────────────────────────
for tag in ENSEMBLE_WEIGHTS:
    if tag == PRIMARY_TAG:
        continue
    tp, op = _load_member(tag)
    if tp is not None:
        member_test[tag] = tp
        if op is not None:
            member_oof[tag] = op
        print(f'[loaded]  {tag}')
    else:
        print(f'[MISSING] {tag} — run `scripts/run_all_10fold.ps1` to regenerate it')

print(f'\nEnsemble members ready: {len(member_test)}/{len(ENSEMBLE_WEIGHTS)} (test), '
      f'{len(member_oof)}/{len(ENSEMBLE_WEIGHTS)} (OOF)')

[CACHED]  primary finbert_fintwitter_10ep_fixtext  (test (2388, 3), oof (9543, 3))
[loaded]  finbert_fintwitter_7ep
[loaded]  roberta_large_ts_v2
[loaded]  debertav3_large_6ep
[loaded]  finbert_fintwitter_10ep
[loaded]  finbert_fintwitter
[loaded]  debertav3_large_6ep_fixtext
[loaded]  deberta_base_finance_fixtext

Ensemble members ready: 8/8 (test), 8/8 (OOF)


In [6]:
# Cell 7: Honest out-of-fold performance of the weighted soft-vote ensemble
# Combines each member's leak-free OOF probabilities with the grid-searched weights.
oof_sum, wsum = np.zeros((len(y), 3)), 0.0
for tag, w in ENSEMBLE_WEIGHTS.items():
    if tag in member_oof:
        oof_sum += member_oof[tag] * w
        wsum += w
oof_ens  = oof_sum / wsum
oof_pred = oof_ens.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

print(f'Ensemble OOF F1-macro : {oof_f1:.4f}   (weight total {wsum:.2f}, '
      f'{len(member_oof)} members)')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))

Ensemble OOF F1-macro : 0.9201   (weight total 6.75, 8 members)
Accuracy     : 0.9357
Precision-mac: 0.9097
Recall-macro : 0.9315

              precision    recall  f1-score   support

     Bearish       0.87      0.92      0.90      1442
     Bullish       0.89      0.93      0.91      1923
     Neutral       0.97      0.94      0.95      6178

    accuracy                           0.94      9543
   macro avg       0.91      0.93      0.92      9543
weighted avg       0.94      0.94      0.94      9543



In [7]:
# Cell 7b: Weighted soft-vote on the test set (the final ensemble prediction)
test_sum, wsum_t = np.zeros((len(test_texts), 3)), 0.0
used = []
for tag, w in ENSEMBLE_WEIGHTS.items():
    if tag in member_test:
        test_sum += member_test[tag] * w
        wsum_t += w
        used.append(f'{tag} (w={w})')
test_proba_final = (test_sum / wsum_t).astype(np.float32)

print(f'Weighted soft-vote over {len(used)} encoders (total weight {wsum_t:.2f}):')
for u in used:
    print(f'  + {u}')
print(f'\nEnsemble OOF F1-macro: {oof_f1:.4f}')

Weighted soft-vote over 8 encoders (total weight 6.75):
  + finbert_fintwitter_10ep_fixtext (w=1.0)
  + finbert_fintwitter_7ep (w=1.0)
  + roberta_large_ts_v2 (w=1.0)
  + debertav3_large_6ep (w=1.0)
  + finbert_fintwitter_10ep (w=0.75)
  + finbert_fintwitter (w=0.75)
  + debertav3_large_6ep_fixtext (w=0.75)
  + deberta_base_finance_fixtext (w=0.5)

Ensemble OOF F1-macro: 0.9201


In [8]:
# Cell 8: Generate predictions and save pred_33.csv (with full validation)
test_pred  = test_proba_final.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)})
submission.to_csv('pred_33.csv', index=False)
os.makedirs('results/predictions', exist_ok=True)
submission.to_csv('results/predictions/pred_final_ensemble.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows) — weighted soft-vote ensemble')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

# Robust validation of the deliverable
assert len(submission) == 2388,                        f'Expected 2388, got {len(submission)}'
assert list(submission.columns) == ['id', 'label'],    'Columns must be exactly [id, label]'
assert set(submission['label'].unique()) == {0, 1, 2}, f'Labels must be {{0,1,2}}'
assert submission['label'].isna().sum() == 0,          'NaN labels found!'
assert submission['id'].is_unique,                     'Duplicate ids found!'
assert (submission['label'] >= 0).all() and (submission['label'] <= 2).all(), 'Labels out of range'
min_class_pct = submission['label'].value_counts(normalize=True).min()
assert min_class_pct > 0.01, f'Under-represented class ({min_class_pct:.1%}) — possible model collapse'

print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))

Predictions saved: pred_33.csv (2388 rows) — weighted soft-vote ensemble
Distribution:
label
Bearish     385
Bullish     497
Neutral    1506
Name: count, dtype: int64

All assertions PASSED.

 id  label
  0      1
  1      2
  2      2
  3      1
  4      2
  5      1
  6      0
  7      0
  8      2
  9      2
